# ORCA QLoRA - Kaggle (stable)

1. Settings -> GPU + Internet On -> **Restart session**
2. Add Input with **train.jsonl**
3. Run cells **1 -> 7 in order**

After any CUDA OOM: Restart session, then start from cell 1 again.


In [ ]:
# ===== 1. Check GPU =====
import gc, sys, torch

def cuda_clean():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print('Python:', sys.version.split()[0])
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Enable GPU in Settings, Save, then Restart session.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 1))
cuda_clean()
print('OK GPU')


In [ ]:
# ===== 2. Install packages =====
import subprocess, sys
pkgs = [
    'transformers==4.51.3', 'peft==0.15.2', 'trl==0.15.2', 'bitsandbytes==0.45.4',
    'accelerate', 'datasets', 'sentencepiece', 'protobuf',
]
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U'] + pkgs, capture_output=True, text=True)
if r.returncode != 0:
    print(r.stderr[-1500:] if r.stderr else r.stdout[-1500:])
    raise RuntimeError('pip install failed')
import transformers, peft, trl, bitsandbytes, datasets
print('transformers', transformers.__version__)
print('Install OK')


In [ ]:
# ===== 3. Find train.jsonl =====
from pathlib import Path

INPUT = Path('/kaggle/input')
if not INPUT.exists():
    raise FileNotFoundError('No /kaggle/input. Add Input dataset with train.jsonl, then Restart.')

jsonl_files = sorted(INPUT.rglob('*.jsonl'))
print('jsonl files:')
for p in jsonl_files:
    print(' ', p)

train_path = None
for p in jsonl_files:
    if p.name.lower() == 'train.jsonl':
        train_path = p
        break
if train_path is None:
    for p in jsonl_files:
        if 'train' in p.name.lower():
            train_path = p
            break
if train_path is None and len(jsonl_files) == 1:
    train_path = jsonl_files[0]

if train_path is None:
    raise FileNotFoundError('train.jsonl not found under /kaggle/input')

n_lines = sum(1 for line in open(train_path, encoding='utf-8') if line.strip())
print('train_path =', train_path)
print('samples =', n_lines)
if n_lines < 1:
    raise ValueError('train file empty')


In [ ]:
# ===== 4. Load + validate messages format =====
from datasets import load_dataset

TRAIN_LIMIT = None  # set 100 if OOM

ds = load_dataset('json', data_files={'train': str(train_path)}, split='train')
row0 = ds[0]
if 'messages' not in row0:
    raise KeyError('Need key messages. Keys=' + str(list(row0.keys())))
msgs = row0['messages']
if not isinstance(msgs, list) or len(msgs) < 2:
    raise ValueError('messages must be a chat list')
print('roles:', [m.get('role') for m in msgs])
print('preview:', str(msgs[-1].get('content', ''))[:160])

if TRAIN_LIMIT is not None:
    ds = ds.select(range(min(int(TRAIN_LIMIT), len(ds))))
print('train size:', len(ds))


In [ ]:
# ===== 5. Model 4-bit + LoRA r=8 =====
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

cuda_clean()
BASE = 'Qwen/Qwen2.5-7B-Instruct'
print('Loading', BASE)

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(BASE, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    BASE, quantization_config=bnb, device_map='auto',
    trust_remote_code=True, low_cpu_mem_usage=True,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

model = get_peft_model(model, LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05, bias='none',
    task_type='CAUSAL_LM', target_modules=['q_proj', 'v_proj'],
))
model.print_trainable_parameters()
try:
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
except TypeError:
    model.gradient_checkpointing_enable()

cuda_clean()
print('VRAM used GB:', round(torch.cuda.memory_allocated() / (1024**3), 2))
print('Model OK')


In [ ]:
# ===== 6. Train =====
import torch
from trl import SFTTrainer, SFTConfig

def formatting_func(example):
    return tokenizer.apply_chat_template(
        example['messages'], tokenize=False, add_generation_prompt=False
    )

MAX_SEQ = 768  # if OOM use 512

train_args = SFTConfig(
    output_dir='/kaggle/working/orca-out',
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=32,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    logging_steps=20,
    save_strategy='epoch',
    eval_strategy='no',
    fp16=True,
    bf16=False,
    optim='paged_adamw_8bit',
    max_seq_length=MAX_SEQ,
    packing=False,
    report_to='none',
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    dataloader_pin_memory=False,
    dataloader_num_workers=0,
    max_grad_norm=0.3,
    group_by_length=True,
    seed=42,
)

cuda_clean()
print('Train', len(ds), 'samples, max_seq', MAX_SEQ)
trainer = SFTTrainer(
    model=model, args=train_args, train_dataset=ds,
    processing_class=tokenizer, formatting_func=formatting_func,
)
try:
    trainer.train()
    print('TRAIN DONE')
except torch.cuda.OutOfMemoryError:
    cuda_clean()
    print('OOM -> Restart session, set TRAIN_LIMIT=100, MAX_SEQ=512, rerun from cell 1')
    raise


In [ ]:
# ===== 7. Save zip =====
from pathlib import Path
import shutil, os, subprocess

out_dir = Path('/kaggle/working/orca-analyst-lora')
if out_dir.exists():
    shutil.rmtree(out_dir)
out_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(out_dir))
tokenizer.save_pretrained(str(out_dir))
print('files:', os.listdir(out_dir)[:8])

zip_path = Path('/kaggle/working/orca-analyst-lora.zip')
if zip_path.exists():
    zip_path.unlink()
subprocess.run(['zip', '-r', 'orca-analyst-lora.zip', 'orca-analyst-lora'], cwd='/kaggle/working', check=True)
print('OK', zip_path, 'MB', round(zip_path.stat().st_size / (1024**2), 1))
print('Download from Working directory / Output')


## Troubleshooting
| Error | Fix |
|-------|-----|
| No CUDA | GPU + Restart |
| train.jsonl not found | Add Input |
| KeyError messages | need chat format |
| CUDA OOM | Restart, TRAIN_LIMIT=100, MAX_SEQ=512 |
